In [ ]:
import datetime 
from dateutil.relativedelta import relativedelta
import pandas as pd
import numpy as np
import db_dtypes
import pymssql
from shutil import copyfile
from openpyxl import load_workbook
import os
from google.cloud import bigquery
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = 'BQ.json'

DB_info = {'server':'192.168.61.119:7622', 'user':'BAReporting', 'password':'KeHeCReme8he'}

client = bigquery.Client()

In [ ]:
sql = f"""
WITH payment_sessions AS (
  SELECT
    DeviceId,
    SessionId,
    MIN(Time) AS first_payment_time
  FROM `openrice-production.ORGA.PV_20260915`
  WHERE DeviceId IS NOT NULL
    AND SessionId IS NOT NULL
    AND (
      LOWER(EventAction) LIKE '%takeaway.pay%'
      OR LOWER(EventLabelRaw) LIKE '%takeaway.pay%'
    )
  GROUP BY DeviceId, SessionId
  ORDER BY first_payment_time
  LIMIT 100
)

SELECT pv.*
FROM `openrice-production.ORGA.PV_20260915` AS pv
INNER JOIN payment_sessions AS payment
  ON pv.DeviceId = payment.DeviceId
  AND pv.SessionId = payment.SessionId
ORDER BY payment.first_payment_time, pv.DeviceId, pv.SessionId, pv.Time;
"""

df_bq = client.query(sql).result().to_dataframe()

In [ ]:
df_bq